# Практическое занятие № 17 — DBSCAN и агломеративная кластеризация
**ТюмГУ · «Технологии искусственного интеллекта — 1» · 90 минут**

Цель: понять ограничения K-means, освоить DBSCAN (ядро / граница / шум) и четыре типа связи в иерархической кластеризации. Параметры выбираются **без целевых меток**. `late` и `actual_minutes` запрещены для кластеризации заказов. Синтетические иллюстрации не являются доказательством качества на реальных данных.

**Материалы:** `orders_dirty.csv` из занятий 3–16 или `demo_orders_17_18.csv` только для проверки запуска. Загружайте файлы в рабочую папку Colab. Все вычисления reproducible при `random_state=42`.

## 1. Ручной пример: ядровая, граничная и шумовая точки (0–15 мин)
DBSCAN использует окрестность $N_\varepsilon(x)=\{x_j:d(x,x_j)\leq\varepsilon\}$, включая саму точку. Ядровая точка: $|N_\varepsilon(x)|\geq MinPts$. Граничная точка достижима из ядра, но сама не является ядром. Шум имеет метку `-1`.

При `eps=0.45`, `min_samples=4` разберите точки B, P и Q вручную (расстояние евклидово).

In [ ]:
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.cluster import DBSCAN, KMeans, AgglomerativeClustering
from sklearn.datasets import make_moons
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import linkage, dendrogram
print('Python:', sys.version.split()[0]); import sklearn; print('scikit-learn:',sklearn.__version__)
np.set_printoptions(precision=3, suppress=True)

POINTS = {'A':(0,0),'B':(.3,0),'C':(0,.3),'D':(.3,.3),
          'E':(3,3),'F':(3.2,3),'G':(3,3.2),'H':(3.2,3.2),
          'P':(.7,.15),'Q':(6,6)}
names=list(POINTS); xy=np.array(list(POINTS.values()),float)
eps_toy=0.45; minpts_toy=4
neighborhood={n:[names[j] for j in np.where(np.linalg.norm(xy-xy[i],axis=1)<=eps_toy+1e-12)[0]] for i,n in enumerate(names)}
for n in ['B','P','Q']:
    print(n, 'N_eps =', neighborhood[n], '|N| =',len(neighborhood[n]))
db_toy=DBSCAN(eps=eps_toy,min_samples=minpts_toy).fit(xy)
core_toy={names[i] for i in db_toy.core_sample_indices_}
classification={n:('шум' if lab==-1 else ('ядро' if n in core_toy else 'граница')) for n,lab in zip(names,db_toy.labels_)}
print('Классификация:',classification)
assert classification['B']=='ядро' and classification['P']=='граница' and classification['Q']=='шум'
fig,ax=plt.subplots(figsize=(7,4.5))
ax.scatter(xy[:,0],xy[:,1],c=db_toy.labels_,cmap='tab10',s=65,edgecolor='black')
for n,(x,y) in POINTS.items():ax.annotate(n,(x+.07,y+.07))
ax.set(xlabel='x',ylabel='y',title='Ручной пример DBSCAN: eps=0.45, MinPts=4');ax.grid(alpha=.2);plt.show()

**Самостоятельно:** объясните, почему P — граница, а не шум. Повторите с `min_samples=3`: как изменится тип P? Не называйте номера кластеров «классами задержки».

## 2. Почему геометрия кластеров важна: две луны (15–30 мин)
На неполусферических группах K-means может разделить данные не по естественным изогнутым структурам. Сравните K-means, DBSCAN и Ward на **одном и том же** искусственном наборе. Для DBSCAN запишите долю шума и число непустых групп.

In [ ]:
X_moon, _ = make_moons(n_samples=240, noise=.075, random_state=42)
rng=np.random.default_rng(42)
outliers=rng.uniform(low=[-1.25,-.95],high=[2.25,1.35],size=(18,2))
X_shape=np.vstack([X_moon,outliers])
shape_models={
    'K-means (K=2)':KMeans(n_clusters=2,n_init=10,random_state=42).fit_predict(X_shape),
    'DBSCAN (eps=0.19)':DBSCAN(eps=.19,min_samples=5).fit_predict(X_shape),
    'Ward (K=2)':AgglomerativeClustering(n_clusters=2,linkage='ward').fit_predict(X_shape)
}
fig,axs=plt.subplots(1,3,figsize=(13,3.9),constrained_layout=True)
for ax,(name,labs) in zip(axs,shape_models.items()):
    ax.scatter(X_shape[:,0],X_shape[:,1],c=labs,cmap='tab10',s=18,alpha=.85)
    ax.set_title(name+'\nгрупп='+str(len(set(labs)-{-1}))+', шум='+str(int(np.sum(labs==-1))))
    ax.set(xlabel='x1',ylabel='x2');ax.grid(alpha=.15)
plt.show()
print('Важно: параметры для примера «луны» нельзя автоматически переносить на заказы.')

## 3. Подготовка заказов и защита от утечки (30–43 мин)
Работаем с тремя доступными при оформлении признаками. Сортируем по `created_at`, 80% ранних заказов — `dev` для выбора настроек. Последние 20% — `holdout`, который **не участвует в подборе `eps` и числа групп**. Заполнение пропусков и стандартизация обучаются только на `dev`. `order_id` нужен для аудита, `late` и `actual_minutes` — лишь для внешнего последующего анализа, не для кластеризации.

In [ ]:
source=Path('orders_dirty.csv')
if not source.exists():source=Path('demo_orders_17_18.csv')
if not source.exists():raise FileNotFoundError('Загрузите orders_dirty.csv или demo_orders_17_18.csv рядом с блокнотом')
raw=pd.read_csv(source,encoding='utf-8-sig')
FEATURES=['distance_km','queue_size','promised_minutes']
required={'order_id','created_at',*FEATURES}
assert required.issubset(raw.columns),f'Нет столбцов: {required-set(raw.columns)}'
print('ИСТОЧНИК:',source.name, '— только демо!' if source.name.startswith('demo') else '— учебный набор')
print('Размер исходника:',raw.shape,'полные дубликаты:',raw.duplicated().sum())
df=raw.drop_duplicates().copy()
df['created_at']=pd.to_datetime(df['created_at'],errors='coerce')
df=df.dropna(subset=['created_at','order_id']).sort_values('created_at').drop_duplicates('order_id',keep='first').reset_index(drop=True)
for col in FEATURES:df[col]=pd.to_numeric(df[col],errors='coerce').replace([np.inf,-np.inf],np.nan)
assert all(c not in FEATURES for c in ['late','actual_minutes','order_id'])
cut=int(len(df)*.8)
assert cut>=20 and len(df)-cut>=5,'Мало валидных заказов для данной схемы'
dev=df.iloc[:cut].copy();holdout=df.iloc[cut:].copy()
assert set(dev['order_id']).isdisjoint(set(holdout['order_id']))
assert dev['created_at'].max()<=holdout['created_at'].min()
imp=SimpleImputer(strategy='median');scale=StandardScaler()
Z_dev=scale.fit_transform(imp.fit_transform(dev[FEATURES]))
Z_holdout=scale.transform(imp.transform(holdout[FEATURES]))
assert np.isfinite(Z_dev).all() and np.isfinite(Z_holdout).all()
print('dev:',len(dev),'holdout:',len(holdout),'признаки:',FEATURES)
print('Медианы (dev):',dict(zip(FEATURES,imp.statistics_.round(3))))
print('Шкала только dev: mean=',np.round(scale.mean_,3))

## 4. DBSCAN: k-distance, подбор eps и ограничение Silhouette (43–63 мин)
Для `min_samples=5` построим расстояние до **пятого соседа, считая саму точку**. Резкий перегиб кривой подсказывает кандидатов `eps`, но не определяет оптимум автоматически. В таблице фиксируем число групп, число/долю шума и Silhouette **только по непризнанным шумом объектам** при наличии хотя бы 2 кластеров. Не сравнивайте этот Silhouette напрямую с оценкой метода без отбрасывания шума.

In [ ]:
min_samples=5
knn=NearestNeighbors(n_neighbors=min_samples).fit(Z_dev)
kdist=np.sort(knn.kneighbors(Z_dev)[0][:,-1]);quantiles=[.65,.80,.90,.95]
eps_candidates=np.unique(np.round(np.quantile(kdist,quantiles),4))
fig,ax=plt.subplots(figsize=(7.5,3.7));ax.plot(np.arange(len(kdist))+1,kdist)
for q in quantiles:ax.axhline(np.quantile(kdist,q),ls='--',lw=.9,label=f'q={q:.2f}')
ax.set(xlabel='Объекты в порядке роста k-distance',ylabel='Расстояние до 5-го соседа',title='Подбор диапазона eps (dev)')
ax.grid(alpha=.2);ax.legend(ncol=2,fontsize=8);plt.show()

def dbscan_metrics(X,labels):
    non_noise=labels!=-1
    ncl=len(set(labels)-{-1});coverage=non_noise.mean()
    sil=np.nan
    if ncl>=2 and non_noise.sum()>ncl:
        sil=silhouette_score(X[non_noise],labels[non_noise])
    return int(ncl),int((~non_noise).sum()),float(coverage),float(sil)

rows=[]; candidates={}
for eps in eps_candidates:
    model=DBSCAN(eps=float(eps),min_samples=min_samples).fit(Z_dev)
    labels=model.labels_; candidates[float(eps)]=model
    ncl,nnoise,coverage,sil=dbscan_metrics(Z_dev,labels)
    rows.append(dict(eps=float(eps),clusters=ncl,noise=nnoise,noise_pct=round(100*(1-coverage),1),silhouette_excl_noise=sil))
results_db=pd.DataFrame(rows)
print(results_db.to_string(index=False))
feasible=results_db[(results_db.clusters>=2)&(results_db.noise_pct<=35)&np.isfinite(results_db.silhouette_excl_noise)]
if len(feasible):
    chosen_eps=float(feasible.sort_values('silhouette_excl_noise',ascending=False).iloc[0].eps)
else:
    chosen_eps=float(eps_candidates[len(eps_candidates)//2]);print('Предупреждение: подходящих кандидатов с >=2 группами и <=35% шума нет; выбран демонстрационный eps.')
chosen_db=candidates[chosen_eps]
print('Выбран eps (только по dev):',chosen_eps,'группы/шум:',dbscan_metrics(Z_dev,chosen_db.labels_)[:2])
fig,ax=plt.subplots(figsize=(7,4));ax.scatter(Z_dev[:,0],Z_dev[:,1],c=chosen_db.labels_,cmap='tab10',s=24,alpha=.85)
ax.set(xlabel='distance_km (z)',ylabel='queue_size (z)',title='DBSCAN: 2D-вид кластеризации в 3D-пространстве')
ax.grid(alpha=.2);plt.show()

**Практическое исследование:** измените `min_samples` на 4 и 8, опишите, как изменились шум и число групп. Если шумовых объектов много, не объявляйте Silhouette «лучшим качеством» без оценки покрытия.

## 5. Иерархическая кластеризация и дендрограмма (63–80 мин)
Сравните `single`, `complete`, `average`, `ward` при одинаковом `K=3`. Ward минимизирует прирост внутрикластерной суммы квадратов; дендрограмма показывает порядок и высоту слияний. В отличие от DBSCAN, `AgglomerativeClustering` не имеет штатного `predict` для новых заказов.

In [ ]:
K=3
rows_h=[];hier_labels={}
for method in ['single','complete','average','ward']:
    labs=AgglomerativeClustering(n_clusters=K,linkage=method).fit_predict(Z_dev)
    hier_labels[method]=labs
    counts=np.bincount(labs)
    sil=silhouette_score(Z_dev,labs) if len(set(labs))>1 and len(set(labs))<len(Z_dev) else np.nan
    rows_h.append(dict(linkage=method,clusters=len(set(labs)),min_size=int(counts.min()),max_size=int(counts.max()),silhouette=sil))
results_h=pd.DataFrame(rows_h);print(results_h.to_string(index=False))
Z_link=linkage(Z_dev,method='ward')
fig,ax=plt.subplots(figsize=(10,4.0))
dendrogram(Z_link,truncate_mode='lastp',p=min(20,len(dev)-1),show_leaf_counts=True,ax=ax)
ax.set(xlabel='Последние узлы дерева (число объектов в скобках)',ylabel='Расстояние объединения (Ward)',title='Усечённая дендрограмма Ward на dev')
plt.xticks(rotation=45,fontsize=8);plt.tight_layout();plt.show()
base=KMeans(n_clusters=K,n_init=10,random_state=42).fit_predict(Z_dev)
print('K-means vs Ward: ARI =',round(adjusted_rand_score(base,hier_labels['ward']),4))

## 6. Вывод и ограничения для будущих заказов (80–90 мин)
DBSCAN не имеет обычного `predict`. Ниже показано **приближённое** правило: присвоить метку ближайшей ядровой точки, если расстояние до неё ≤ `eps`, иначе шум. Оно не пересчитывает плотность и не является полноценным DBSCAN на новых данных. Обязательно опишите этот компромисс в отчёте.

In [ ]:
core_idx=chosen_db.core_sample_indices_
if len(core_idx):
    nearest_core=NearestNeighbors(n_neighbors=1).fit(Z_dev[core_idx])
    distances,idx=nearest_core.kneighbors(Z_holdout)
    approx_labels=np.where(distances[:,0]<=chosen_eps,chosen_db.labels_[core_idx[idx[:,0]]],-1)
else:
    approx_labels=np.full(len(holdout),-1,dtype=int)
print('Holdout: условно присвоено групп',len(set(approx_labels)-{-1}),'отмечено шумом',np.sum(approx_labels==-1))
exports=holdout[['order_id','created_at']].copy();exports['dbscan_nearest_core_approx']=approx_labels
exports.to_csv('17_holdout_approx.csv',index=False)
results_db.to_csv('17_dbscan_metrics.csv',index=False)
results_h.to_csv('17_hierarchical_metrics.csv',index=False)
print('Созданы 17_holdout_approx.csv, 17_dbscan_metrics.csv, 17_hierarchical_metrics.csv')
assert len(exports)==len(holdout) and exports.order_id.is_unique

### Что сдавать
1. Отчёт 2–4 страницы (без приложений): метод, исходная и рабочая выборки, ручной пример B/P/Q, две таблицы метрик, **k-distance**, дендрограмма, интерпретация 2–3 групп, ограничения/риск утечек.
2. Исполняемый `17_student.ipynb` (Restart & Run All) и по возможности CSV с результатами.
3. **Самостоятельно:** сравните результаты DBSCAN при `min_samples=4,5,8`; объясните, почему разные условия масштабирования изменяют понятие плотности.

**Контроль:** в каких ситуациях K-means, Ward и DBSCAN принципиально расходятся? Что означает `-1`? Почему `silhouette` при высоком шуме может вводить в заблуждение?